In [4]:
import os
os.getcwd()

'/home/philipp/deng/deng-ingestion-pipeline/notebooks/raw_checks'

In [5]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# ------------------------------------------------------------
# Locate lookup files
# ------------------------------------------------------------
lookup_dir = Path("../../data/lookups/")
lookup_files = sorted(lookup_dir.glob("*.txt"))

print(f"Found {len(lookup_files)} lookup files:")
for file in lookup_files:
    print(f" - {file.name}")

# ------------------------------------------------------------
# Simple delimiter detection
# ------------------------------------------------------------
def detect_delimiter(file_path: Path) -> str:
    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            tab_count = line.count("\t")
            comma_count = line.count(",")
            semicolon_count = line.count(";")

            if tab_count >= comma_count and tab_count >= semicolon_count and tab_count > 0:
                return "\t"
            if comma_count >= semicolon_count and comma_count > 0:
                return ","
            if semicolon_count > 0:
                return ";"

            # fallback: whitespace
            return r"\s+"

    return r"\s+"

# ------------------------------------------------------------
# Preview each lookup file
# ------------------------------------------------------------
lookup_frames = {}

for file in lookup_files:
    display(Markdown(f"## {file.name}"))

    # Show first raw lines
    print("First 5 raw lines:")
    with open(file, "r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            print(repr(line.rstrip("\n")))
            if i >= 4:
                break

    sep = detect_delimiter(file)
    print(f"\nDetected separator: {sep!r}")

    try:
        if sep == r"\s+":
            df_lookup = pd.read_csv(
                file,
                sep=sep,
                header=None,
                dtype=str,
                engine="python",
            )
        else:
            df_lookup = pd.read_csv(
                file,
                sep=sep,
                header=None,
                dtype=str,
            )

        lookup_frames[file.name] = df_lookup

        print(f"Shape: {df_lookup.shape}")
        display(df_lookup.head())

    except Exception as exc:
        print(f"Could not parse {file.name}: {exc}")

Found 8 lookup files:
 - CAMEO.country.txt
 - CAMEO.ethnic.txt
 - CAMEO.eventcodes.txt
 - CAMEO.goldsteinscale.txt
 - CAMEO.knowngroup.txt
 - CAMEO.religion.txt
 - CAMEO.type.txt
 - FIPS.country.txt


## CAMEO.country.txt

First 5 raw lines:
'CODE\tLABEL'
'WSB\tWest Bank'
'BAG\tBaghdad'
'GZS\tGaza Strip'
'AFR\tAfrica'

Detected separator: '\t'
Shape: (262, 2)


,0,1
0,CODE,LABEL
1,WSB,West Bank
2,BAG,Baghdad
3,GZS,Gaza Strip
4,AFR,Africa


## CAMEO.ethnic.txt

...

First 5 raw lines:
'CODE\tLABEL'
'COP\tPolice forces'
'GOV\tGovernment'
'INS\tInsurgents'
'JUD\tJudiciary'

Detected separator: '\t'
Shape: (41, 2)


,0,1
0,CODE,LABEL
1,COP,Police forces
2,GOV,Government
3,INS,Insurgents
4,JUD,Judiciary


## FIPS.country.txt

First 5 raw lines:
'AF\tAfghanistan'
'AX\tAkrotiri Sovereign Base Area'
'AL\tAlbania'
'AG\tAlgeria'
'AQ\tAmerican Samoa'

Detected separator: '\t'
Shape: (274, 2)


,0,1
0,AF,Afghanistan
1,AX,Akrotiri Sovereign Base Area
2,AL,Albania
3,AG,Algeria
4,AQ,American Samoa


In [11]:
file = lookup_dir / "CAMEO.eventcodes.txt"

df_eventcodes = pd.read_csv(
    file,
    sep="\t",
    header=0,
    dtype=str,
)

print(df_eventcodes.shape)
df_eventcodes = df_eventcodes.rename(
    columns={
        "CAMEOEVENTCODE": "event_code",
        "EVENTDESCRIPTION": "event_description",
    }
)

display(df_eventcodes.head(20))

print(df_eventcodes.info())
print(df_eventcodes["event_code"].head(20).to_list())
print(df_eventcodes["event_code"].duplicated().sum())

(310, 2)


,event_code,event_description
0,01,MAKE PUBLIC STATEMENT
1,010,"Make statement, not specified below"
2,011,Decline comment
3,012,Make pessimistic comment
4,013,Make optimistic comment
5,014,Consider policy option
6,015,Acknowledge or claim responsibility
7,016,Deny responsibility
8,017,Engage in symbolic act
9,018,Make empathetic comment


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 310 entries, 0 to 309
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   event_code         310 non-null    object
 1   event_description  310 non-null    object
dtypes: object(2)
memory usage: 5.0+ KB
None
['01', '010', '011', '012', '013', '014', '015', '016', '017', '018', '019', '02', '020', '021', '0211', '0212', '0213', '0214', '022', '023']
0


In [28]:
df_cameo_country = df_cameo_country_raw.copy()

# Use first row as header
df_cameo_country.columns = df_cameo_country.iloc[0]
df_cameo_country = df_cameo_country.iloc[1:].reset_index(drop=True)

print("Cleaned shape:")
print(df_cameo_country.shape)

print("Columns:")
print(df_cameo_country.columns.to_list())

df_cameo_country = df_cameo_country.rename(
    columns={
        "CODE": "country_code",
        "LABEL": "country_name",
    }
)

print(df_cameo_country.info())

print("\nDuplicate country_code count:")
print(df_cameo_country["country_code"].duplicated().sum())

print("\nNull counts:")
print(df_cameo_country.isna().sum())

print("\nCode length distribution:")
print(df_cameo_country["country_code"].astype(str).str.len().value_counts().sort_index())

print("\nSample sorted codes:")
display(df_cameo_country.sort_values("country_code").head(30))

Cleaned shape:
(261, 2)
Columns:
['CODE', 'LABEL']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 261 entries, 0 to 260
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   country_code  261 non-null    object
 1   country_name  261 non-null    object
dtypes: object(2)
memory usage: 4.2+ KB
None

Duplicate country_code count:
0

Null counts:
0
country_code    0
country_name    0
dtype: int64

Code length distribution:
country_code
3    261
Name: count, dtype: int64

Sample sorted codes:


,country_code,country_name
39,ABW,Aruba
28,AFG,Afghanistan
3,AFR,Africa
34,AGO,Angola
35,AIA,Anguilla
29,ALA,Aland Islands
30,ALB,Albania
33,AND,Andorra
176,ANT,Netherlands Antilles
247,ARE,United Arab Emirates


In [35]:
display(Markdown("## Analyze FIPS.country.txt"))

df_fips_country = lookup_frames["FIPS.country.txt"].copy()

print("Raw shape:")
print(df_fips_country.shape)

df_fips_country.columns = ["country_code", "country_name"]

print(df_fips_country.info())

print("\nDuplicate country_code count:")
print(df_fips_country["country_code"].duplicated().sum())

print("\nNull counts:")
print(df_fips_country.isna().sum())

print("\nCode length distribution:")
print(df_fips_country["country_code"].astype(str).str.len().value_counts().sort_index())

print("\nSample sorted codes:")
display(df_fips_country.sort_values("country_code").head(30))

## Analyze FIPS.country.txt

Raw shape:
(274, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 274 entries, 0 to 273
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   country_code  274 non-null    object
 1   country_name  274 non-null    object
dtypes: object(2)
memory usage: 4.4+ KB
None

Duplicate country_code count:
0

Null counts:
country_code    0
country_name    0
dtype: int64

Code length distribution:
country_code
2    274
Name: count, dtype: int64

Sample sorted codes:


,country_code,country_name
12,AA,Aruba
9,AC,Antigua and Barbuda
257,AE,United Arab Emirates
0,AF,Afghanistan
3,AG,Algeria
16,AJ,Azerbaijan
2,AL,Albania
11,AM,Armenia
5,AN,Andorra
6,AO,Angola


In [38]:
display(Markdown("## Analyze CAMEO.goldsteinscale.txt"))

df_goldstein = lookup_frames["CAMEO.goldsteinscale.txt"].copy()

df_goldstein.columns = df_goldstein.iloc[0]
df_goldstein = df_goldstein.iloc[1:].reset_index(drop=True)
df_goldstein.columns = df_goldstein.columns.astype(str).str.strip()

df_goldstein = df_goldstein.rename(
    columns={
        "CAMEOEVENTCODE": "event_code",
        "GOLDSTEINSCALE": "goldstein_scale",
    }
)

print(df_goldstein.info())

print("\nDuplicate event_code count:")
print(df_goldstein["event_code"].duplicated().sum())

print("\nNull counts:")
print(df_goldstein.isna().sum())

print("\nCode length distribution:")
print(df_goldstein["event_code"].astype(str).str.len().value_counts().sort_index())

goldstein_series = df_goldstein["goldstein_scale"].dropna().astype(str).str.strip()
goldstein_series = goldstein_series[goldstein_series != ""]

decimal_lengths = goldstein_series.apply(
    lambda x: len(x.split(".")[1]) if "." in x else 0
)

numeric_series = pd.to_numeric(goldstein_series, errors="coerce")

print("\nMin:", numeric_series.min())
print("Max:", numeric_series.max())
print("Max decimal places:", decimal_lengths.max())

print("\nDistribution of decimal places:")
print(decimal_lengths.value_counts().sort_index())

print("\nSample values:")
print(goldstein_series.drop_duplicates().sort_values().head(30).to_list())

## Analyze CAMEO.goldsteinscale.txt

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 310 entries, 0 to 309
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   event_code       310 non-null    object
 1   goldstein_scale  310 non-null    object
dtypes: object(2)
memory usage: 5.0+ KB
None

Duplicate event_code count:
0

Null counts:
0
event_code         0
goldstein_scale    0
dtype: int64

Code length distribution:
event_code
2     20
3    146
4    144
Name: count, dtype: int64

Min: -10.0
Max: 10.0
Max decimal places: 1

Distribution of decimal places:
goldstein_scale
1    310
Name: count, dtype: int64

Sample values:
['-0.1', '-0.3', '-0.4', '-10.0', '-2.0', '-4.0', '-4.4', '-5.0', '-5.6', '-5.8', '-6.0', '-6.5', '-7.0', '-7.2', '-7.5', '-8.0', '-9.0', '-9.2', '-9.5', '0.0', '0.4', '1.0', '1.9', '10.0', '2.5', '2.8', '3.0', '3.2', '3.4', '3.5']


In [43]:
df_cameo_event_codes = df_eventcodes.copy()

df_cameo_event_codes["event_base_code"] = df_cameo_event_codes["event_code"].apply(
    lambda x: x[:3] if len(x) == 4 else x
)
df_cameo_event_codes["event_root_code"] = df_cameo_event_codes["event_code"].apply(
    lambda x: x[:2]
)

df_cameo_event_codes = df_cameo_event_codes.merge(
    df_goldstein,
    on="event_code",
    how="left",
    validate="one_to_one",
)

df_cameo_event_codes["goldstein_scale"] = pd.to_numeric(
    df_cameo_event_codes["goldstein_scale"],
    errors="raise",
)

display(df_cameo_event_codes.head(20))
print(df_cameo_event_codes.info())

,event_code,event_description,event_base_code,event_root_code,goldstein_scale
0,01,MAKE PUBLIC STATEMENT,01,01,0.0
1,010,"Make statement, not specified below",010,01,0.0
2,011,Decline comment,011,01,-0.1
3,012,Make pessimistic comment,012,01,-0.4
4,013,Make optimistic comment,013,01,0.4
5,014,Consider policy option,014,01,0.0
6,015,Acknowledge or claim responsibility,015,01,0.0
7,016,Deny responsibility,016,01,-2.0
8,017,Engage in symbolic act,017,01,0.0
9,018,Make empathetic comment,018,01,3.4


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 310 entries, 0 to 309
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   event_code         310 non-null    object 
 1   event_description  310 non-null    object 
 2   event_base_code    310 non-null    object 
 3   event_root_code    310 non-null    object 
 4   goldstein_scale    310 non-null    float64
dtypes: float64(1), object(4)
memory usage: 12.2+ KB
None


In [40]:
print(df_cameo_event_codes["goldstein_scale"].isna().sum())

0


In [53]:
display(Markdown("## Analyze CAMEO.knowngroup.txt"))

df_known_group = lookup_frames["CAMEO.knowngroup.txt"].copy()

print("Raw shape:")
print(df_known_group.shape)

df_known_group.columns = df_known_group.iloc[0]
df_known_group = df_known_group.iloc[1:].reset_index(drop=True)
df_known_group.columns = df_known_group.columns.astype(str).str.strip()

print("Columns:")
print(df_known_group.columns.to_list())

df_known_group = df_known_group.rename(
    columns={
        "CODE": "known_group_code",
        "LABEL": "known_group_name",
    }
)

print(df_known_group.info())

print("\nDuplicate known_group_code count:")
print(df_known_group["known_group_code"].duplicated().sum())

print("\nNull counts:")
print(df_known_group.isna().sum())

print("\nCode length distribution:")
print(df_known_group["known_group_code"].astype(str).str.len().value_counts().sort_index())

print("\nEmpty-string counts:")
print((df_known_group.astype(str).apply(lambda col: col.str.strip() == "")).sum())

print("\nSample sorted codes:")
display(df_known_group.sort_values("known_group_code").head(30))

## Analyze CAMEO.knowngroup.txt

Raw shape:
(118, 2)
Columns:
['CODE', 'LABEL']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   known_group_code  117 non-null    object
 1   known_group_name  117 non-null    object
dtypes: object(2)
memory usage: 2.0+ KB
None

Duplicate known_group_code count:
2

Null counts:
0
known_group_code    0
known_group_name    0
dtype: int64

Code length distribution:
known_group_code
3    117
Name: count, dtype: int64

Empty-string counts:
0
known_group_code    0
known_group_name    0
dtype: int64

Sample sorted codes:


,known_group_code,known_group_name
0,AAM,Al Aqsa Martyrs Brigade
1,ABD,Arab Bank for Economic Development in Africa
2,ACC,Arab Cooperation Council
3,ADB,Asian Development Bank
4,AEU,Arab Economic Unity Council
5,AFB,African Development Bank
6,ALQ,Al Qaeda
7,AMF,Arab Monetary Fund for Economic and Social Dev...
8,AML,Amal Militia
9,AMN,Amnesty International


In [54]:
duplicates = df_known_group[
    df_known_group["known_group_code"].duplicated(keep=False)
].sort_values("known_group_code")

display(duplicates)

,known_group_code,known_group_name
20,CEM,Common Market for Eastern and Southern Africa
21,CEM,Monetary and Economic Community of Central Africa
114,WTO,World Trade Organization
115,WTO,World Trade Organization (WTO)


In [55]:
print(df_known_group["known_group_code"].duplicated().sum())

2


In [58]:
df_known_group_curated = df_known_group.copy()

df_known_group_curated.loc[
    df_known_group_curated["known_group_code"] == "WTO",
    "known_group_name"
] = "World Trade Organization"

df_known_group_curated.loc[
    df_known_group_curated["known_group_code"] == "CEM",
    "known_group_name"
] = "AMBIGUOUS: CEMAC / COMESA"

df_known_group_curated = df_known_group_curated.drop_duplicates(
    subset=["known_group_code"],
    keep="first",
).reset_index(drop=True)

df_known_group_curated["is_ambiguous"] = False
df_known_group_curated.loc[
    df_known_group_curated["known_group_code"] == "CEM",
    "is_ambiguous"
] = True

print(df_known_group_curated["known_group_code"].duplicated().sum())
display(df_known_group_curated[df_known_group_curated["known_group_code"] == "CEM"])
display(df_known_group_curated[df_known_group_curated["known_group_code"] == "WTO"])

0


,known_group_code,known_group_name,is_ambiguous
20,CEM,AMBIGUOUS: CEMAC / COMESA,True


,known_group_code,known_group_name,is_ambiguous
113,WTO,World Trade Organization,False
